# 03 — IRT Calibration & Adaptive Testing

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Agent:** DiagnosticAgent (IRT 3PL + CAT)  
**Purpose:** Calibrate item parameters, validate ability estimates,
and generate publication-quality figures for the paper.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

print("Libraries loaded.")

## 1. Load Data & Calibrate IRT

In [ ]:
from data.loader import EdNetLoader
from agents.diagnostic_agent import DiagnosticAgent

loader = EdNetLoader(data_dir="../data/raw")
questions = loader.load_questions()
interactions = loader.load_interactions(sample_users=1000)

print(f"Interactions: {len(interactions):,} rows, {interactions['user_id'].nunique()} users")
print(f"Unique questions in interactions: {interactions['question_id'].nunique()}")

In [ ]:
# Calibrate IRT from interactions
diag = DiagnosticAgent()
params = diag.calibrate_from_interactions(
    interactions,
    min_answers_per_q=30,
    max_items=5000,
)

print(f"\nCalibrated {len(params)} items")
print(f"  b (difficulty):      mean={params.b.mean():.3f}, std={params.b.std():.3f}, range=[{params.b.min():.2f}, {params.b.max():.2f}]")
print(f"  a (discrimination):  mean={params.a.mean():.3f}, std={params.a.std():.3f}, range=[{params.a.min():.2f}, {params.a.max():.2f}]")
print(f"  c (guessing):        {params.c[0]:.2f} (fixed)")

## 2. Difficulty Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Difficulty histogram
axes[0].hist(params.b, bins=50, color="#3498db", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", alpha=0.7, label="b=0 (medium)")
axes[0].set_xlabel("Difficulty (b)")
axes[0].set_ylabel("Number of Items")
axes[0].set_title("Distribution of Item Difficulty")
axes[0].legend()

# Difficulty by part
part_data = pd.DataFrame({"b": params.b, "part": params.part_ids})
part_means = part_data.groupby("part")["b"].mean().sort_index()
bars = axes[1].bar(part_means.index.astype(str), part_means.values,
                    color="#e74c3c", edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, part_means.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f"{val:.2f}", ha="center", fontsize=9)
axes[1].set_xlabel("TOEIC Part")
axes[1].set_ylabel("Mean Difficulty (b)")
axes[1].set_title("Average Item Difficulty by TOEIC Part")
axes[1].axhline(0, color="gray", linestyle=":", alpha=0.5)

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_irt_difficulty.png")
plt.show()

## 3. Discrimination Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(params.a, bins=50, color="#2ecc71", edgecolor="white", linewidth=0.3)
ax.axvline(params.a.mean(), color="red", linestyle="--",
           label=f"Mean: {params.a.mean():.2f}")
ax.set_xlabel("Discrimination (a)")
ax.set_ylabel("Number of Items")
ax.set_title("Distribution of Item Discrimination")
ax.legend()
sns.despine()

fig.savefig(RESULTS_DIR / "fig_irt_discrimination.png")
plt.show()

# Stats
print(f"Low discrimination (a < 0.5): {(params.a < 0.5).sum()} items")
print(f"High discrimination (a > 1.5): {(params.a > 1.5).sum()} items")

## 4. Item Characteristic Curves (ICC)

In [ ]:
thetas = np.linspace(-4, 4, 200)

# Select 5 representative items: easy, medium-easy, medium, medium-hard, hard
b_sorted = np.argsort(params.b)
n = len(params)
sample_indices = [
    b_sorted[n // 10],          # easy
    b_sorted[n // 4],           # medium-easy
    b_sorted[n // 2],           # medium
    b_sorted[3 * n // 4],       # medium-hard
    b_sorted[9 * n // 10],      # hard
]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c", "#8e44ad"]
labels = ["Easy", "Medium-Easy", "Medium", "Medium-Hard", "Hard"]

for idx, color, label in zip(sample_indices, colors, labels):
    icc = diag.icc_curve(idx, thetas)
    qid = params.question_ids[idx]
    ax.plot(thetas, icc, color=color, linewidth=2,
            label=f"{label}: b={params.b[idx]:.2f}, a={params.a[idx]:.2f}")

ax.axhline(0.25, color="gray", linestyle=":", alpha=0.5, label="Guessing (c=0.25)")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel(r"Ability ($\theta$)")
ax.set_ylabel(r"P(correct | $\theta$)")
ax.set_title("Item Characteristic Curves (3PL)")
ax.legend(fontsize=9, loc="lower right")
ax.set_ylim(-0.02, 1.02)
sns.despine()

fig.savefig(RESULTS_DIR / "fig_icc_curves.png")
plt.show()

## 5. Test Information Function

In [ ]:
tif = diag.test_information_function(thetas)
se_curve = 1.0 / np.sqrt(np.clip(tif, 1e-10, None))

fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = "#3498db"
ax1.plot(thetas, tif, color=color1, linewidth=2, label="Test Information")
ax1.set_xlabel(r"Ability ($\theta$)")
ax1.set_ylabel("Test Information I(\u03b8)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
color2 = "#e74c3c"
ax2.plot(thetas, se_curve, color=color2, linewidth=2, linestyle="--", label="Standard Error")
ax2.set_ylabel("Standard Error SE(\u03b8)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)
ax2.set_ylim(0, 2)

ax1.set_title("Test Information Function & Standard Error")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

fig.savefig(RESULTS_DIR / "fig_test_information.png")
plt.show()

# Peak information
peak_theta = thetas[np.argmax(tif)]
print(f"Peak information at theta={peak_theta:.2f}, I={tif.max():.1f}")
print(f"SE at peak: {se_curve[np.argmax(tif)]:.4f}")

## 6. Estimate Abilities & Validate Against Accuracy

In [ ]:
# Estimate theta for each user from their responses
user_ids = interactions["user_id"].unique()
user_stats = []

for uid in user_ids:
    user_df = interactions[interactions["user_id"] == uid]
    result = diag.run_assessment(uid, user_df)
    actual_acc = user_df["correct"].mean()
    user_stats.append({
        "user_id": uid,
        "theta": result["ability"],
        "se": result["se"],
        "accuracy": actual_acc,
        "n_responses": len(user_df),
    })

user_df = pd.DataFrame(user_stats)
print(f"Estimated abilities for {len(user_df)} users")
print(user_df[["theta", "se", "accuracy"]].describe().round(3))

In [ ]:
# Scatter: theta vs actual accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Theta vs Accuracy
ax = axes[0]
sc = ax.scatter(user_df["theta"], user_df["accuracy"],
                c=user_df["n_responses"], cmap="viridis",
                s=20, alpha=0.6, edgecolors="none")
plt.colorbar(sc, ax=ax, label="N responses")

# Correlation
r_pearson, p_val = pearsonr(user_df["theta"], user_df["accuracy"])
r_spearman, _ = spearmanr(user_df["theta"], user_df["accuracy"])

# Fit line
z = np.polyfit(user_df["theta"], user_df["accuracy"], 1)
poly = np.poly1d(z)
x_line = np.linspace(user_df["theta"].min(), user_df["theta"].max(), 100)
ax.plot(x_line, poly(x_line), "r--", linewidth=1.5)

ax.set_xlabel(r"Estimated Ability ($\theta$)")
ax.set_ylabel("Actual Accuracy")
ax.set_title(f"IRT Ability vs Accuracy (r={r_pearson:.3f}, p={p_val:.1e})")
ax.text(0.05, 0.95, f"Pearson r = {r_pearson:.3f}\nSpearman \u03c1 = {r_spearman:.3f}",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# SE distribution
ax = axes[1]
ax.hist(user_df["se"], bins=40, color="#9b59b6", edgecolor="white", linewidth=0.3)
ax.axvline(0.3, color="red", linestyle="--", label="CAT threshold (0.3)")
ax.set_xlabel("Standard Error of Ability")
ax.set_ylabel("Number of Users")
ax.set_title("Distribution of Ability Estimation SE")
ax.legend()

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_theta_vs_accuracy.png")
plt.show()

## 7. CAT Simulation

In [ ]:
# Simulate CAT for a few users
sample_users = user_ids[:5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

convergence_items = []

for uid in sample_users:
    user_interactions = interactions[interactions["user_id"] == uid]
    # Build simulated response dict
    sim_resp = {}
    for _, row in user_interactions.iterrows():
        qid = str(row["question_id"])
        sim_resp[qid] = bool(row["correct"])

    cat_result = diag.run_cat_session(
        user_id=f"cat_{uid}",
        simulated_responses=sim_resp,
        max_items=20,
    )

    # Plot theta convergence
    steps = range(1, len(cat_result["responses"]) + 1)
    thetas_seq = [r["theta_after"] for r in cat_result["responses"]]
    se_seq = [r["se_after"] for r in cat_result["responses"]]

    axes[0].plot(steps, thetas_seq, marker="o", markersize=3, label=f"User {uid}")
    axes[1].plot(steps, se_seq, marker="o", markersize=3, label=f"User {uid}")
    convergence_items.append(cat_result["n_items"])

axes[0].set_xlabel("Number of Items")
axes[0].set_ylabel(r"$\theta$ Estimate")
axes[0].set_title("CAT: Ability Convergence")
axes[0].legend(fontsize=8)

axes[1].axhline(0.3, color="red", linestyle="--", alpha=0.7, label="Threshold")
axes[1].set_xlabel("Number of Items")
axes[1].set_ylabel("Standard Error")
axes[1].set_title("CAT: SE Convergence")
axes[1].legend(fontsize=8)

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_cat_convergence.png")
plt.show()

print(f"Average items to convergence: {np.mean(convergence_items):.1f}")

## 8. Diagnostic Test Demo

In [ ]:
# Run diagnostic for a sample user
uid = user_ids[0]
user_responses = interactions[interactions["user_id"] == uid]
sim_resp = {str(r["question_id"]): bool(r["correct"]) for _, r in user_responses.iterrows()}

diag_result = diag.run_diagnostic(f"demo_{uid}", simulated_responses=sim_resp)

print(f"Diagnostic Result for user {uid}:")
print(f"  Ability (theta): {diag_result['ability']}")
print(f"  SE: {diag_result['se']}")
print(f"  Items: {diag_result['n_questions']}")
print(f"  Parts covered: {diag_result['parts_covered']}")
print(f"\nPer-item details:")
for r in diag_result["responses"]:
    mark = "correct" if r["correct"] else "wrong"
    print(f"  {r['question_id']:>8s}  part={r['part_id']}  b={r['difficulty']:+.2f}  {mark}")

In [ ]:
print("\n=== IRT Calibration & CAT Complete ===")
print(f"Items calibrated: {len(params)}")
print(f"Users assessed: {len(user_df)}")
print(f"Theta-accuracy correlation: r={r_pearson:.3f}")
print(f"Figures saved to: {RESULTS_DIR.resolve()}")